# 第 7 章 · 多 agent（LoopAgent）

一个干活的 agent + 一个检查状态的自定义 agent，循环到**条件满足**或撞 `max_iterations`。

⚠️ 书里那版永远停不下来：instruction 让模型"把 status 设成 completed"，
**但 LlmAgent 写不了 state**——它只能"说"完成了，状态纹丝不动。下面用工具补上。

★ 本版把"完成任务"做成**随机事件**（每轮 `DONE_PROB` 概率成功），所以能真正看到
循环跑多轮、以及"条件停"和"预算停"的区别。

★ 循环**全部跑完之后**再做一次总结：先打一份确定性的轮次统计（轮数 / 尝试次数 /
停止方式 / 逐轮记录），再把统计交给一个 loop **外**的 `Reporter` agent 写成两三句话。
这样"统计"和"总结"分开：数字不花 token，叙述才调模型。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

try:
    from google.adk.runners import InMemoryRunner
    from google.genai import types
except ModuleNotFoundError:
    # 【适配】内核选错时的提示；比裸 ModuleNotFoundError 好定位
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state


import json
import random
from typing import AsyncGenerator

from google.adk.agents import BaseAgent, LlmAgent, LoopAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event, EventActions
from google.adk.tools import ToolContext
from google.genai import types

DONE_PROB = 0.4          # ★ 每轮"完成任务"的概率；调低能看到更多轮


def try_finish(tool_context: ToolContext) -> str:
    """尝试完成当前步骤。成功返回 DONE 并把 status 置为 completed；否则返回 NOT_DONE。"""
    # 【适配】书里让模型直接改 state —— LlmAgent 做不到，只能通过工具写
    n = tool_context.state.get("attempts", 0) + 1
    tool_context.state["attempts"] = n
    ok = random.random() < DONE_PROB
    tool_context.state.setdefault("log", []).append(
        f"try_finish#{n} -> {'DONE' if ok else 'NOT_DONE'}")
    if ok:
        tool_context.state["status"] = "completed"
        return "DONE"
    return "NOT_DONE: 本轮没做成，还有问题没解决"


class ConditionChecker(BaseAgent):
    """读 state 里的 status 决定要不要停。"""
    name: str = "ConditionChecker"
    description: str = "Checks if a process is complete and signals the loop to stop."

    async def _run_async_impl(self, context: InvocationContext) -> AsyncGenerator[Event, None]:
        status = context.session.state.get("status", "pending")
        if status == "completed":
            # ★ 本章新增 escalate：置为 True 就是"跳出循环"的信号
            #   （第 4 章那个 exit_loop 工具，内部做的就是这件事）
            yield Event(author=self.name, actions=EventActions(escalate=True))
        else:
            yield Event(author=self.name,
                        content=types.Content(role="model", parts=[types.Part(text="继续下一轮")]))


worker = LlmAgent(
    name="Worker",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="每一轮你必须先调用 try_finish 工具（这是你唯一的动作），"
                "再根据它的返回值说一句话：DONE 就说“已完成”，否则说“还需要继续”。"
                "不要只说话而不调用工具。",
    tools=[try_finish],
)

MAX_ROUNDS = 6
poller = LoopAgent(name="Poller", max_iterations=MAX_ROUNDS,
                   sub_agents=[worker, ConditionChecker()])


def run_loop(seed):
    """跑一次循环，返回本轮运行的统计。按 ConditionChecker 的执行次数计轮数。"""
    random.seed(seed)
    runner = InMemoryRunner(poller)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    rounds = 0
    for ev in runner.run(user_id="u", session_id=sid,
                         new_message=types.Content(role="user", parts=[types.Part(text="完成这个任务")])):
        if getattr(ev, "author", "") == "ConditionChecker":
            rounds += 1
    s = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid).state
    status = s.get("status")
    return {
        "seed": seed,
        "rounds": rounds,
        "attempts": s.get("attempts", 0),
        "final_status": status,
        "stopped_by": "条件停（escalate）" if status == "completed" else "预算停（max_iterations）",
        "log": list(s.get("log") or []),
    }


def print_stats(st):
    """把一次运行的统计打出来 —— 这部分是确定性的，不花 token。"""
    idle = st["rounds"] - st["attempts"]          # 该轮没调工具的轮数
    print(f"  [轮次统计] 轮数={st['rounds']}  尝试={st['attempts']}  "
          f"未调工具轮={idle}  最终状态={st['final_status']!r}  停止方式={st['stopped_by']}")
    print("  逐轮记录:")
    for i, line in enumerate(st["log"], 1):
        print(f"    {i}. {line}")


# ★ 循环跑完之后，再让一个 agent 做一次【总结】——它不在 loop 里，只被调用一次
reporter = LlmAgent(
    name="Reporter",
    model=adk_model(),
    instruction="下面是一次循环任务的运行统计。用两三句中文总结：跑了几轮、每轮结果如何、最终是否成功。",
)

for seed in (2, 4):
    print(f"\n===== seed={seed} =====")
    st = run_loop(seed)
    print_stats(st)
    finals, _ = run_once(reporter, json.dumps(st, ensure_ascii=False, default=str))
    print(f"  [总结] {finals[-1][1] if finals else '(无)'}")



===== seed=2 =====


  [轮次统计] 轮数=3  尝试=3  未调工具轮=0  最终状态='completed'  停止方式=条件停（escalate）
  逐轮记录:
    1. try_finish#1 -> NOT_DONE
    2. try_finish#2 -> NOT_DONE
    3. try_finish#3 -> DONE


  [总结] 本次任务共进行了3轮尝试。前两轮执行均未完成（NOT_DONE），在第三轮中成功达成目标（DONE）。最终，任务因触发条件停（escalate）而顺利完成。

===== seed=4 =====


  [轮次统计] 轮数=1  尝试=1  未调工具轮=0  最终状态='completed'  停止方式=条件停（escalate）
  逐轮记录:
    1. try_finish#1 -> DONE


  [总结] 本次任务共运行了1轮。在该轮尝试中，系统通过触发“条件停”成功结束了运行。最终，任务顺利完成。
